# LangChain Guardrails - Practical Colab Guide

Based on the [LangChain Guardrails docs](https://docs.langchain.com/oss/python/langchain/guardrails)

**Guardrails** protect your AI app by intercepting content at key points.

The docs cover two approaches:
- **Deterministic** - fast regex/keyword rules (no extra LLM cost)
- **Model-based** - a second LLM judges the content (catches subtle issues)

| Scenario | Guardrail | What it prevents |
|----------|-----------|-----------------|
| 1. Hospital chatbot | PII Middleware | Patients leaking emails and card numbers |
| 2. Banking agent | Human-in-the-Loop | AI executing wire transfers without approval |
| 3. Cybersecurity helpdesk | before_agent | Staff asking about offensive hacking |
| 4. Kids homework helper | after_agent | LLM returning age-inappropriate content |
| 5. HR assistant | All layers | Everything above in one production stack |


---
## Step 1 - Install and Setup

Run this cell first. If you hit import errors later, come back and run it again.


In [3]:
# Install packages
!pip install -q langchain langchain-openai langgraph

import os, re
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage

# Paste your OpenAI API key here
os.environ["OPENAI_API_KEY"] = "sk-proj--"

# LLM used in every scenario
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# --- Shared print helpers ---
def show_ok(label, text):
    print(f"  OK       [{label}]  {str(text)[:100]}")

def show_blocked(label, text):
    print(f"  BLOCKED  [{label}]  {str(text)[:100]}")

def show_modified(label, before, after):
    print(f"  MODIFIED [{label}]")
    print(f"    Before: {str(before)[:70]}")
    print(f"    After : {str(after)[:70]}")

def section(title):
    print(f"\n{'='*55}\n  {title}\n{'='*55}")

print("Ready!")


Ready!


---
## Scenario 1 - Hospital Patient Intake Chatbot
### Guardrail: PIIMiddleware

**Problem:** Patients type medical questions in plain text and often include sensitive info.

The docs define 4 strategies for handling PII:

| Strategy | What happens | Example |
|----------|-------------|---------|
| redact | Full replacement | `john@gmail.com` becomes `[REDACTED_EMAIL]` |
| mask | Partial hide | `5105-1051-0510-5100` becomes `****-****-****-5100` |
| hash | Deterministic hash | `a8f5f167...` |
| block | Raise error immediately | Exception thrown |

We guard **both input** (before LLM) and **output** (before patient sees it).


In [4]:
# Scenario 1 - PII Middleware
# Implements: redact / mask / block strategies from the docs

# --- REDACT: full replacement ---
def redact_emails(text):
    return re.sub(r'[\w\.-]+@[\w\.-]+\.\w+', '[REDACTED_EMAIL]', text)

# --- MASK: show only last 4 digits ---
def mask_credit_cards(text):
    def _mask(m):
        digits = re.sub(r'\D', '', m.group())
        return f"****-****-****-{digits[-4:]}"
    return re.sub(r'\b(?:\d[ -]?){13,16}\b', _mask, text)

# --- BLOCK: raise immediately, request goes no further ---
def block_api_keys(text):
    if re.search(r'sk-[a-zA-Z0-9]{20,}', text):
        raise ValueError("API key detected - request blocked for security")


# --- The guarded chatbot ---
def hospital_chatbot(user_message):
    # INPUT GUARDRAILS (before LLM sees anything)
    block_api_keys(user_message)
    clean = redact_emails(user_message)
    clean = mask_credit_cards(clean)

    if clean != user_message:
        show_modified("PII sanitized", user_message, clean)

    # LLM call - only sees the cleaned message
    response = llm.invoke([HumanMessage(content=
        "You are a helpful hospital chatbot. Answer the patient briefly.\n"
        f"Patient: {clean}"
    )]).content

    # OUTPUT GUARDRAILS (scrub anything that leaked into the response)
    response = redact_emails(response)
    response = mask_credit_cards(response)
    return response


# --- Tests ---
section("Scenario 1 - Hospital PII Guardrail")

tests = [
    ("Normal query",  "What are the visiting hours for Ward B?"),
    ("Has email",     "Please send my results to jane.doe@gmail.com"),
    ("Has card",      "I want to pay - my card is 5105-1051-0510-5100"),
    ("Has API key",   "Log me in with sk-abcdefghij1234567890abcdefghijk"),
]

for label, msg in tests:
    print(f"\n  Patient: {msg[:65]}")
    try:
        reply = hospital_chatbot(msg)
        show_ok(label, reply)
    except ValueError as e:
        show_blocked(label, e)



  Scenario 1 - Hospital PII Guardrail

  Patient: What are the visiting hours for Ward B?
  OK       [Normal query]  Visiting hours for Ward B are from 2 PM to 8 PM daily. Please check with the ward for any specific r

  Patient: Please send my results to jane.doe@gmail.com
  MODIFIED [PII sanitized]
    Before: Please send my results to jane.doe@gmail.com
    After : Please send my results to [REDACTED_EMAIL]
  OK       [Has email]  I'm sorry, but I can't send results via email. Please contact your healthcare provider directly for 

  Patient: I want to pay - my card is 5105-1051-0510-5100
  MODIFIED [PII sanitized]
    Before: I want to pay - my card is 5105-1051-0510-5100
    After : I want to pay - my card is ****-****-****-5100
  OK       [Has card]  I'm sorry, but I can't process payments or handle sensitive information like credit card numbers. Pl

  Patient: Log me in with sk-abcdefghij1234567890abcdefghijk
  BLOCKED  [Has API key]  API key detected - request blocked for secur

---
## Scenario 2 - Banking Transaction Agent
### Guardrail: HumanInTheLoopMiddleware

**Problem:** An AI agent can trigger wire transfers and close accounts. These are irreversible - a human must approve first.

From the docs:
```python
HumanInTheLoopMiddleware(interrupt_on={
    "wire_transfer": True,   # pause, wait for approval
    "get_balance":   False,  # auto-proceed (read-only)
})
```

After pausing, the compliance officer resumes with:
```python
agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=...)
```


In [5]:
# Scenario 2 - Human-in-the-Loop
# Pattern: detect sensitive action -> pause -> human decides -> resume or abort

# --- Simulated bank tools ---
def get_balance(account_id):
    return f"Balance for {account_id}: $12,450.00"

def get_history(account_id):
    return f"Last 3 txns for {account_id}: -$120 (grocery), +$3,000 (salary), -$45 (utilities)"

def wire_transfer(to_account, amount):
    return f"Wire of ${amount:,.2f} to {to_account} executed."

def close_account(account_id):
    return f"Account {account_id} has been closed."


SENSITIVE_ACTIONS = {"wire_transfer", "close_account"}

def banking_agent(user_request, human_decision="approve"):
    """
    Simulates HumanInTheLoopMiddleware:
      1. Determine what action the user wants
      2. If sensitive: pause and wait for human decision
      3. If approved: execute. If rejected: abort.
      4. If safe (read-only): execute immediately.
    """
    req = user_request.lower()

    # Simple intent detection
    if "balance" in req:
        action, fn, args = "get_balance", get_balance, ("ACC-001",)
    elif "history" in req or "transaction" in req:
        action, fn, args = "get_history", get_history, ("ACC-001",)
    elif "wire" in req or "transfer" in req or "send" in req:
        amt = re.search(r'\$?([\d,]+)', user_request)
        amount = float(amt.group(1).replace(',','')) if amt else 1000.0
        action, fn, args = "wire_transfer", wire_transfer, ("EXT-9924", amount)
    elif "close" in req:
        action, fn, args = "close_account", close_account, ("ACC-001",)
    else:
        return llm.invoke([HumanMessage(content=f"Banking question: {user_request}")]).content

    # HumanInTheLoop check
    if action in SENSITIVE_ACTIONS:
        print(f"  PAUSED - '{action}' requires human approval")
        print(f"  Compliance Officer decision: {human_decision.upper()}")
        if human_decision != "approve":
            return "Action rejected. No changes made."

    return fn(*args)


# --- Tests ---
section("Scenario 2 - Banking Human-in-the-Loop")

tests = [
    ("approve", "What is my current balance?"),
    ("approve", "Show my recent transactions"),
    ("approve", "Send $5,000 to account EXT-9924"),     # HITL - approved
    ("reject",  "Close my account ACC-001 immediately"), # HITL - rejected
]

for decision, msg in tests:
    print(f"\n  Request: {msg}")
    r = banking_agent(msg, human_decision=decision)
    if decision == "reject":
        show_blocked("Result", r)
    else:
        show_ok("Result", r)



  Scenario 2 - Banking Human-in-the-Loop

  Request: What is my current balance?
  OK       [Result]  Balance for ACC-001: $12,450.00

  Request: Show my recent transactions
  OK       [Result]  Last 3 txns for ACC-001: -$120 (grocery), +$3,000 (salary), -$45 (utilities)

  Request: Send $5,000 to account EXT-9924
  PAUSED - 'wire_transfer' requires human approval
  Compliance Officer decision: APPROVE
  OK       [Result]  Wire of $5,000.00 to EXT-9924 executed.

  Request: Close my account ACC-001 immediately
  PAUSED - 'close_account' requires human approval
  Compliance Officer decision: REJECT
  BLOCKED  [Result]  Action rejected. No changes made.


---
## Scenario 3 - Cybersecurity Helpdesk Bot
### Guardrail: `before_agent`

**Problem:** Offensive security questions must be caught **before the LLM is called** - zero token cost.

The docs show two equivalent syntaxes:

**Class syntax** - when you need configurable `__init__` parameters:
```python
class ContentFilterMiddleware(AgentMiddleware):
    @hook_config(can_jump_to=["end"])
    def before_agent(self, state, runtime): ...
```

**Decorator syntax** - simpler for stateless filters:
```python
@before_agent(can_jump_to=["end"])
def content_filter(state, runtime): ...
```

Both do the exact same thing. We implement both so you can compare.


In [6]:
# Scenario 3 - before_agent guardrail
# Runs BEFORE the LLM is called - fast, no API cost

BANNED_KEYWORDS = [
    "exploit", "ransomware", "ddos", "sql injection",
    "zero-day", "rootkit", "keylogger", "privilege escalation"
]

BLOCKED_MSG = (
    "This request violates our acceptable use policy. "
    "Contact the security team if you have a legitimate need."
)


# ==========================================================
# A: CLASS SYNTAX
# Good when your filter needs __init__ configuration
# ==========================================================
class ContentFilterMiddleware:
    def __init__(self, banned_keywords):
        self.banned = [kw.lower() for kw in banned_keywords]

    def before_agent(self, message):
        """Returns a block message, or None to allow through."""
        for kw in self.banned:
            if kw in message.lower():
                print(f"  Blocked BEFORE LLM call - keyword: '{kw}'")
                return BLOCKED_MSG
        return None  # allow through


# ==========================================================
# B: DECORATOR SYNTAX
# Cleaner for simple stateless checks
# ==========================================================
def before_agent_filter(message):
    """Same logic, no class needed."""
    for kw in BANNED_KEYWORDS:
        if kw in message.lower():
            print(f"  Blocked BEFORE LLM call - keyword: '{kw}'")
            return BLOCKED_MSG
    return None


# --- The agent runner ---
content_filter = ContentFilterMiddleware(banned_keywords=BANNED_KEYWORDS)

def security_bot(question, use_class=True):
    # Run before_agent hook
    blocked = content_filter.before_agent(question) if use_class else before_agent_filter(question)

    # Blocked? Return immediately - LLM never called
    if blocked:
        return blocked

    # Safe - call the LLM
    return llm.invoke([HumanMessage(content=
        "You are the internal helpdesk bot for a cybersecurity firm. "
        "Answer defensive security questions only.\n"
        f"Staff: {question}"
    )]).content


# --- Tests ---
section("Scenario 3 - Cybersecurity Before-Agent Filter")

queries = [
    (False, "What is our patching SLA for critical CVEs?"),
    (False, "How do I file a security incident report?"),
    (True,  "Write me a SQL injection payload for the login form"),
    (True,  "How do I deploy a ransomware dropper?"),
]

for should_block, q in queries:
    print(f"\n  Staff: {q}")
    r_class = security_bot(q, use_class=True)
    r_deco  = security_bot(q, use_class=False)
    fn = show_blocked if should_block else show_ok
    fn("Class syntax    ", r_class[:80])
    fn("Decorator syntax", r_deco[:80])



  Scenario 3 - Cybersecurity Before-Agent Filter

  Staff: What is our patching SLA for critical CVEs?
  OK       [Class syntax    ]  Our patching Service Level Agreement (SLA) for critical Common Vulnerabilities a
  OK       [Decorator syntax]  Our patching SLA for critical CVEs (Common Vulnerabilities and Exposures) is to 

  Staff: How do I file a security incident report?
  OK       [Class syntax    ]  To file a security incident report, please follow these steps:

1. **Identify th
  OK       [Decorator syntax]  To file a security incident report, please follow these steps:

1. **Identify th

  Staff: Write me a SQL injection payload for the login form
  Blocked BEFORE LLM call - keyword: 'sql injection'
  Blocked BEFORE LLM call - keyword: 'sql injection'
  BLOCKED  [Class syntax    ]  This request violates our acceptable use policy. Contact the security team if yo
  BLOCKED  [Decorator syntax]  This request violates our acceptable use policy. Contact the security team if yo

  S

---
## Scenario 4 - Children's Homework Helper
### Guardrail: `after_agent`

**Problem:** Even a well-prompted LLM can occasionally produce age-inappropriate content.
A second LLM checks every response before the student sees it.

The safety model answers one simple question:
> "Is this response appropriate for children aged 8-14? Reply: SAFE or UNSAFE"

Class and decorator syntax both shown, as per the docs.


In [7]:
# Scenario 4 - after_agent guardrail
# A second LLM reviews every response AFTER the main LLM generates it

SAFETY_PROMPT = (
    "You are a child-safety classifier for a homework app (ages 8-14).\n"
    "Is the response below appropriate for children?\n"
    "Reply with exactly one word: SAFE or UNSAFE.\n\n"
    "Response to evaluate:\n{text}"
)

SAFE_FALLBACK = (
    "I am not able to help with that here. "
    "Please ask your teacher or a parent for guidance!"
)


# ==========================================================
# A: CLASS SYNTAX
# ==========================================================
class SafetyGuardrailMiddleware:
    def __init__(self):
        # A cheap fast model works well as a safety judge
        self.safety_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    def after_agent(self, response):
        """Returns original if safe, fallback if flagged."""
        verdict = self.safety_model.invoke([
            HumanMessage(content=SAFETY_PROMPT.format(text=response))
        ]).content.strip().upper()

        print(f"  Safety model verdict: {verdict}")

        if "UNSAFE" in verdict:
            return SAFE_FALLBACK
        return response


# ==========================================================
# B: DECORATOR SYNTAX
# ==========================================================
_safety_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def after_agent_safety(response):
    """Same logic, no class needed."""
    verdict = _safety_llm.invoke([
        HumanMessage(content=SAFETY_PROMPT.format(text=response))
    ]).content.strip().upper()

    print(f"  Safety model verdict: {verdict}")

    if "UNSAFE" in verdict:
        return SAFE_FALLBACK
    return response


# --- The agent runner ---
safety_check = SafetyGuardrailMiddleware()

def homework_bot(question, use_class=True):
    # Step 1: Main LLM generates a response
    raw = llm.invoke([HumanMessage(content=
        f"You are a homework helper for kids aged 8-14. Answer: {question}"
    )]).content

    # Step 2: after_agent hook checks before the student sees it
    return safety_check.after_agent(raw) if use_class else after_agent_safety(raw)


# --- Tests ---
section("Scenario 4 - Kids Homework After-Agent Safety Check")

questions = [
    (False, "What is photosynthesis?"),
    (False, "How do volcanoes form?"),
    (True,  "Describe violent historical battles in graphic detail"),
]

for should_flag, q in questions:
    print(f"\n  Student: {q}")
    r = homework_bot(q, use_class=True)
    fn = show_blocked if should_flag else show_ok
    fn("Response", r[:90])



  Scenario 4 - Kids Homework After-Agent Safety Check

  Student: What is photosynthesis?
  Safety model verdict: SAFE
  OK       [Response]  Photosynthesis is the process that plants, algae, and some bacteria use to make their own 

  Student: How do volcanoes form?
  Safety model verdict: SAFE
  OK       [Response]  Volcanoes form when molten rock, called magma, from deep inside the Earth rises to the sur

  Student: Describe violent historical battles in graphic detail
  Safety model verdict: SAFE
  BLOCKED  [Response]  While it's important to understand history, especially significant events like battles, it


---
## Scenario 5 - Enterprise HR Assistant
### All layers combined

This is what a real production stack looks like.
Each layer catches what the previous one missed.

```
Incoming message
    |
    v  Layer 1: Content filter   (before, fast, no LLM cost)
    |
    v  Layer 2: PII sanitize     (before, redact emails and cards)
    |
    v  LLM + Tools
    |
    v  Layer 3: Human approval   (pause on sensitive data mutations)
    |
    v  Layer 4: Safety check     (after, model-based)
    |
  Safe, approved response
```

From the docs:
```python
agent = create_agent(
    model="gpt-4o-mini",
    middleware=[
        ContentFilterMiddleware(...),   # Layer 1: deterministic, before
        PIIMiddleware(...),             # Layer 2: sanitize
        HumanInTheLoopMiddleware(...),  # Layer 3: approve
        SafetyGuardrailMiddleware(),    # Layer 4: model-based, after
    ],
)
```


In [8]:
# Scenario 5 - All guardrails combined
# Reuses everything from Scenarios 1-4

# --- Simulated HR tools ---
def get_employee_record(emp_id):
    data = {
        "EMP-001": "Alice Chen | Engineering | Salary: $95,000",
        "EMP-002": "Bob Smith  | Engineering | Salary: $110,000",
    }
    return data.get(emp_id, f"No record found for {emp_id}")

def update_salary(emp_id, new_salary):
    return f"Salary for {emp_id} updated to ${new_salary:,.2f}"

def delete_employee(emp_id):
    return f"Employee {emp_id} removed from the system."


HR_BANNED    = ["discriminat", "harass", "illegal", "falsif"]
HR_SENSITIVE = {"update_salary", "delete_employee"}

def hr_assistant(user_message, auto_approve=True):
    print(f"\n  Received: '{user_message[:60]}'")

    # === LAYER 1: Content filter (before_agent) ===
    for kw in HR_BANNED:
        if kw in user_message.lower():
            print(f"  Layer 1: BLOCKED - keyword '{kw}'")
            return "This request violates HR policy and cannot be processed."

    # === LAYER 2: PII sanitize ===
    clean = redact_emails(user_message)
    clean = mask_credit_cards(clean)
    if clean != user_message:
        print("  Layer 2: PII detected and sanitized")

    # --- Route to the right tool ---
    msg = clean.lower()

    if "record" in msg or "info" in msg:
        emp = re.search(r'EMP-\d+', user_message)
        emp_id = emp.group() if emp else "EMP-001"
        result = get_employee_record(emp_id)

    elif "salary" in msg and any(w in msg for w in ["update","change","raise","increase"]):
        emp   = re.search(r'EMP-\d+', user_message)
        emp_id = emp.group() if emp else "EMP-001"
        amt   = re.search(r'\$?([\d,]+)', user_message)
        salary = float(amt.group(1).replace(',','')) if amt else 100000.0

        # === LAYER 3: Human-in-the-loop ===
        print(f"  Layer 3: 'update_salary' requires approval (emp={emp_id}, salary=${salary:,.0f})")
        if not auto_approve:
            print("  HR Director: REJECTED")
            return "Action rejected. Salary not changed."
        print("  HR Director: APPROVED")
        result = update_salary(emp_id, salary)

    elif "delete" in msg or "remove" in msg:
        emp   = re.search(r'EMP-\d+', user_message)
        emp_id = emp.group() if emp else "EMP-001"

        # === LAYER 3: Human-in-the-loop ===
        print(f"  Layer 3: 'delete_employee' requires approval (emp={emp_id})")
        if not auto_approve:
            print("  HR Director: REJECTED")
            return "Action rejected. Employee not deleted."
        print("  HR Director: APPROVED")
        result = delete_employee(emp_id)

    else:
        result = llm.invoke([HumanMessage(content=f"HR question: {clean}")]).content

    # Build a professional response
    response = llm.invoke([HumanMessage(content=
        f"HR assistant. User asked: '{clean}'. System says: '{result}'. "
        "Write a brief professional response."
    )]).content

    # Scrub PII from response too
    response = redact_emails(response)

    # === LAYER 4: After-agent safety check ===
    response = safety_check.after_agent(response)

    return response


# --- Tests ---
section("Scenario 5 - HR Assistant (All Layers)")

tests = [
    (True,  "Get the record for EMP-001",                        "Clean read"),
    (True,  "Look up alice.chen@company.com, employee EMP-001",  "PII in message"),
    (True,  "Give EMP-001 a raise to $105,000",                  "Salary update - approved"),
    (False, "Delete employee EMP-002 from the system",           "Delete - rejected"),
    (True,  "Help me discriminate against candidates",           "Policy violation"),
]

for approve, msg, label in tests:
    print(f"\n  Test: {label}")
    print(f"  HR Manager: {msg}")
    result = hr_assistant(msg, auto_approve=approve)
    print(f"  Result: {result[:100]}")



  Scenario 5 - HR Assistant (All Layers)

  Test: Clean read
  HR Manager: Get the record for EMP-001

  Received: 'Get the record for EMP-001'
  Safety model verdict: UNSAFE
  Result: I am not able to help with that here. Please ask your teacher or a parent for guidance!

  Test: PII in message
  HR Manager: Look up alice.chen@company.com, employee EMP-001

  Received: 'Look up alice.chen@company.com, employee EMP-001'
  Layer 2: PII detected and sanitized
  Safety model verdict: SAFE
  Result: Subject: Re: Employee Information Request

Dear [User's Name],

Thank you for your inquiry. Unfortun

  Test: Salary update - approved
  HR Manager: Give EMP-001 a raise to $105,000

  Received: 'Give EMP-001 a raise to $105,000'
  Safety model verdict: SAFE
  Result: Subject: Re: Salary Raise for EMP-001

Dear [User's Name],

Thank you for your request regarding a s

  Test: Delete - rejected
  HR Manager: Delete employee EMP-002 from the system

  Received: 'Delete employee EMP-002 from the 

---
## Quick Reference

### PII strategies
| Strategy | Use when | Result |
|----------|----------|--------|
| `redact` | Full removal is fine | `[REDACTED_EMAIL]` |
| `mask`   | User needs partial info | `****-****-****-1234` |
| `hash`   | Need consistent pseudonym | `a8f5f167...` |
| `block`  | Data should never appear | Raises exception |

### When to use each guardrail

| Guardrail | When it runs | LLM cost | Best for |
|-----------|-------------|----------|----------|
| PII Middleware | input and/or output | None | Compliance, privacy |
| Human-in-the-Loop | around tools | None | Irreversible actions |
| `before_agent` | before LLM call | None | Keyword/pattern blocking |
| `after_agent` | after LLM call | +1 LLM call | Subtle safety issues |

### The golden rule
> Put cheap deterministic checks first.
> Put expensive model-based checks last.
